In [10]:
import os
from skimage.metrics import structural_similarity
from skimage.metrics import mean_squared_error
import math
import numpy as np
import sirf.Gadgetron as pMR
import matplotlib.pyplot as plt


data_path = '/home/jovyan/work/SIRF-Contribs/src/notebooks/'
os.chdir(data_path)
import utils
from utils import plot_rpe_3d, plot_rpe_3d_simple

# Can use either of these two as ground truth
# ground_truth_with_TV = np.load(os.path.join(data_path, 'ground_truth_with_TV.npy'))
# ground_truth_no_TV = np.load(os.path.join(data_path, 'ground_truth_no_TV.npy'))


Nms = 3
file_pattern = r'{}_L2_ms_{:d}_it_{:03d}.h5'

crop = 20


recons = []
# basedir = 'recons_ignore-acq_4bis'
basedir = 'recons_ignore-acq_equal-tv'
# for ms in range(Nms):
#     _fname = os.path.join(data_path, basedir, f'MS_{Nms}', file_pattern.format('FISTA', ms, 20) )
#     recons.append( pMR.ImageData(file=_fname).as_array()[crop:-crop,:,:] )




In [7]:
# NEW FUNCTION for plotting with yellow line

def plot_rpe_3d_with_yellow_line(dat, sl_idx, lbl, cmap='gray', wspace=-.32, hspace=0.02):
    hspace=0.06

    fig, ax = plt.subplots(1, len(dat), squeeze=True, figsize=(12,3), gridspec_kw={'hspace':hspace})
    print(ax)
    
    for ind in range(len(dat)):       
        sp = ax[ind].imshow(np.rot90(np.abs(dat[ind][:, :, sl_idx[1]])), cmap=cmap)
        ax[ind].set_xticks([])
        ax[ind].set_yticks([])
        ax[ind].set_title(lbl[ind])
        
        #set xlabels
        ax[ind].set_xlabel('Right-Left')
        
    ax[0].set_ylabel('Anterior-Posterior')
        
    # add vertical line
    pos=60-20
    x = np.linspace(pos, pos, 2)
    y = np.linspace(20, 108, 2)
    color="yellow"
    line = "--"
    ax[0].plot(x,y,line,color=color)
    ax[1].plot(x,y,line,color=color)
    ax[2].plot(x,y,line,color=color)
    # ax[1,0].plot(x,y,line,color=color)
    # ax[1,1].plot(x,y,line,color=color)
    # ax[1,2].plot(x,y,line,color=color)
    # plt.tight_layout()
    plt.savefig('motion_state_example.png', bbox_inches='tight')
    plt.show()


In [8]:
import importlib
# increase the font size
# https://matplotlib.org/stable/users/explain/customizing.html
import matplotlib
# import matplotlib.pylab as pylab
# params = {'axes.titlesize':'16', 'axes.labelsize':'16'}

matplotlib.rcParams.update(matplotlib.rcParamsDefault)
matplotlib.rcParams.update({'font.size': 14})
# pylab.rcParams.update(params)
importlib.reload(utils)

slice_index = [64, 64]
plot_rpe_3d_with_yellow_line(recons, slice_index * 3, [f'MS {i}' for i in range(Nms)])


ValueError: Number of columns must be a positive integer, not 0

<Figure size 1200x300 with 0 Axes>

In [9]:
recons[0].shape

IndexError: list index out of range

In [10]:
recons2 = {}
file_pattern2 = r'{}_TV_60ms_it_{:03d}.h5'

for epochs in [10, 70]:
    for algo in ['FISTA', 'PDHG', 'SPDHG']:
        iters = epochs
        if algo == 'SPDHG':
            iters *= 60
        _fname = os.path.join(data_path, 'recons_ignore-acq_4bis', f'MS_60', file_pattern2.format(algo, iters) )
        recons2[algo+'_'+str(epochs)] = pMR.ImageData(file=_fname).as_array()[crop:-crop,:,:]


# reference image
algo = 'PDHG'
epochs = 200
iters = epochs
_fname = os.path.join(data_path, 'recons_ignore-acq_4bis', f'MS_60', file_pattern2.format(algo, iters) )
recons2[algo+'_'+str(epochs)] = pMR.ImageData(file=_fname).as_array()[crop:-crop,:,:]


In [11]:
def select_slice_ant_pos(data, idx):
    return np.rot90(np.abs(data[:, :, idx]), 1) #np.rot90(np.abs(np.abs(data[idx, :, :]), 1) )
    # return np.rot90(np.abs(data[64, :, :]), 1)

def save_fig(fig, title):
    fig.savefig(os.path.join(data_path+'/single_pics_for_10_70_figure', title), bbox_inches='tight')
    return 

In [12]:
def create_single_fig(matrix, title):
    fig, ax = plt.subplots(1, 1 , squeeze=True, figsize=(4.5, 3.4)) #(5, 4.5))#
    im = ax.imshow(matrix, vmax = 3*10**(-5),  cmap='gray')
    fig.suptitle(' '.join(title.split('_')) + ' epochs', fontsize = 18)
    # ax.set_xticks([])
    # ax.set_yticks([])
    ax.axis('off')
    
    # cbar = fig.colorbar(im, ax=ax, orientation='vertical', shrink = 0.9)
    # cbar.ax.tick_params(labelsize=18)
    # cbar.ax.yaxis.get_offset_text().set_fontsize(18)

    return fig


In [6]:
slices = {}
for title, matrix in recons2.items():
    slice_i = select_slice_ant_pos(data=matrix, idx=64)
    slices[title] = slice_i
#     save_fig(create_fig(slice_i, title), title)
    

In [7]:
print(recons2.keys())

dict_keys([])


In [8]:
def create_3x2_fig(slices_dict, cmap='gray', wspace=0.02, hspace=0.06):

    fig, ax = plt.subplots(2, 3, squeeze=True, figsize=(12,7), gridspec_kw={'hspace':hspace})
    ax[0,0].imshow(slices_dict['SPDHG_10'], cmap=cmap)
    ax[0,1].imshow(slices_dict['PDHG_10'], cmap=cmap)
    ax[0,2].imshow(slices_dict['FISTA_10'], cmap=cmap)

    ax[1,0].imshow(slices_dict['SPDHG_70'], cmap=cmap)
    ax[1,1].imshow(slices_dict['PDHG_70'], cmap=cmap)
    ax[1,2].imshow(slices_dict['FISTA_70'], cmap=cmap)

    # set titles
    ax[0,0].title.set_text('SPDHG 10 epochs')
    ax[0,1].title.set_text('PDHG 10 epochs')
    ax[0,2].title.set_text('FISTA 10 epochs')
    
    ax[1,0].title.set_text('SPDHG 70 epochs')
    ax[1,1].title.set_text('PDHG 70 epochs')
    ax[1,2].title.set_text('FISTA 70 epochs')
    
    for i in range(2):
        for j in range(3):
            ax[i, j].axis('off')

    plt.savefig('NEW_phantom_recons_10vs70.png', bbox_inches='tight')
    plt.show()

    

In [9]:
create_3x2_fig(slices)

NameError: name 'plt' is not defined